In [ ]:
import pandas as pd
import numpy as np
from jax import numpy as jnp
from scipy.optimize import minimize
from jax import jit
import matplotlib.pyplot as plt
import diffrax as dfx

from summer3.epi import CompartmentalModelODE, CategoryData, strat_data_from_pandas, build_istate, dti_to_epoch

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA, MAX_AGE, DATA_PATH, ISO3, START_TIME, END_TIME, YOUNG_END_AGE
from tb_macro.epi import get_base_model, add_natural_history, add_seeding, add_latency_flows, add_infection_flows
from tb_macro.inputs import (
    get_country_pop,
    get_single_age_pop_from_ungroups,
    get_group_popsizes,
    get_un_mortality,
    add_groups_to_single_pop,
    build_age_weight_lookup,
    get_fertility_data,
    calc_tsr_from_outcomes,
    calc_death_in_unsucc_outcomes,
    get_country_indicators,
)
from tb_macro.demography import add_replacement_deaths, add_ageing_flows, prepare_pop_data_for_entries, add_entry_births
from tb_macro.health_system import add_treatment_flows, add_detection
from tb_macro.parameters import BASE_PARAMS
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET
from tb_macro.calibration import make_log_likelihood
from tb_macro.plotting import plot_comp_distributions, plot_dynamic_mixing_matrix

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Data loading and processing
pop_data = get_country_pop(ISO3)
single_age_pops = get_single_age_pop_from_ungroups(pop_data)
group_popsize = get_group_popsizes(single_age_pops)
mort_data = get_un_mortality(ISO3)
death_rates = mort_data.div(group_popsize, axis=0).dropna()
add_groups_to_single_pop(single_age_pops)
age_weights = build_age_weight_lookup(single_age_pops)
fert = get_fertility_data(ISO3)
fert_padded = fert.reindex(columns=range(MAX_AGE + 1), fill_value=0.0)
raw_outcome_data = pd.read_csv(DATA_PATH / "who/who_outcomes_20260514T0437Z.csv")
outcome_data = raw_outcome_data[raw_outcome_data["iso3"] == ISO3]
tsr = calc_tsr_from_outcomes(outcome_data)
death_in_unsucc = calc_death_in_unsucc_outcomes(outcome_data)
who_indicators = get_country_indicators(ISO3)
who_mort = who_indicators["e_mort_tbhiv_num"] + who_indicators["e_mort_exc_tbhiv_num"]

In [ ]:
# Model construction
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))

add_infection_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat, age_weights, group_popsize, fert_padded, YOUNG_END_AGE, START_TIME)
add_natural_history(epi_model, disease_state, age_strat, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state, START_TIME)
add_detection(epi_model, disease_state, clin_strat, START_TIME)
add_replacement_deaths(epi_model, disease_state, age_strat, death_rates, START_TIME)
add_entry_births(epi_model, disease_state, age_strat, START_TIME, entry_rates, entry_times)
add_treatment_flows(death_rates, START_TIME, epi_model, disease_state, age_strat, infect_strat, clin_strat, tsr, death_in_unsucc)
add_latency_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat)

# Initialisation
init_apops_series = pd.Series(index=[str(a) for a in AGE_STRATA], data=np.array(start_apops))
init_apops = strat_data_from_pandas(init_apops_series, age_strat)
init_dpops = [0.0] * len(ALL_COMPARTMENTS)
init_dpops[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_dpops)))]
epi_model.set_initial_population(init_apops, pop_splits)
epi_model.computed_values.append("dynamic_mm")

In [ ]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
runner, istate = get_runner(epi_model)

In [ ]:
solver_kwargs = {
    "max_steps": 4000,
    "stepsize_controller": dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0),
    "adjoint": dfx.RecursiveCheckpointAdjoint(2048),
}

In [ ]:
latent_date = LATENT_TARGET.index[0]
latent_target_val = LATENT_TARGET.iloc[0] / 1e2

calib_params = ["contact_rate", "detect_val_2"]


def vector_to_params(calib_params, x):
    return dict(zip(calib_params, x))


def params_to_vector(calib_params, params):
    return np.array([params[p] for p in calib_params])


In [ ]:
# Optimisation
log_like = make_log_likelihood(epi_model, disease_state, solver_kwargs, latent_date, latent_target_val, NOTIF_TARGET, who_mort)

@jit
def opt_cr(x):
    return -log_like(BASE_PARAMS | vector_to_params(calib_params, x))

opt_res = minimize(opt_cr, params_to_vector(calib_params, BASE_PARAMS), method="Nelder-Mead")
opt_res.x

In [ ]:
# Optimisation results
opt_param = vector_to_params(calib_params, opt_res.x)
results = epi_model.run(BASE_PARAMS | opt_param, solver_kwargs=solver_kwargs)

In [ ]:
plot_comp_distributions(results, disease_state, age_strat, infect_strat, clin_strat, END_TIME, group_popsize)

In [ ]:
plot_dynamic_mixing_matrix(results["computed_values"]["dynamic_mm"], 1970.0, 10.0, 3)